# Trader Sentiment and Behavior Analysis
This notebook analyzes how Bitcoin market sentiment (Fear/Greed) affects trader performance and behavior using Hyperliquid trade data.


In [492]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [493]:
# load CSVs
sentiment_df= pd.read_csv('csv_files/fear_greed_index.csv')
trades_df= pd.read_csv('csv_files/historical_data.csv')

# preview data
display(sentiment_df.head())
display(trades_df.head())


,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


#### Clean and prepare datasets

In [ ]:
# ----------------------------
#  Clean Sentiment Dataset
# ----------------------------


# Convert timestamp (UNIX seconds) → datetime
if np.issubdtype(sentiment_df['timestamp'].dtype, np.number):
    sentiment_df['timestamp'] = pd.to_datetime(sentiment_df['timestamp'], unit='s', errors='coerce')
else:
    sentiment_df['timestamp'] = pd.to_datetime(sentiment_df['timestamp'], errors='coerce')


sentiment_df['date']=pd.to_datetime(sentiment_df['date'], errors='coerce').dt.date

sentiment_df['time_only'] = sentiment_df['timestamp'].dt.time

sentiment_df['classification']=sentiment_df['classification'].astype(str).str.strip().str.title()



# Check for missing values
print("Null values in Sentiment Dataset:")
print(sentiment_df.isna().sum())

# Remove duplicates (by date)
sentiment_df.drop_duplicates(subset=['date'], inplace=True)

# Reset index
sentiment_df.reset_index(drop=True, inplace=True)

# Print summary
print("✅ Sentiment dataset cleaned.")
print(sentiment_df.info())
display(sentiment_df.head())

Null values in Sentiment Dataset:
timestamp         0
value             0
classification    0
date              0
time_only         0
dtype: int64
✅ Sentiment dataset cleaned.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   timestamp       2644 non-null   datetime64[ns]
 1   value           2644 non-null   int64         
 2   classification  2644 non-null   object        
 3   date            2644 non-null   object        
 4   time_only       2644 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 103.4+ KB
None


,timestamp,value,classification,date,time_only
0,2018-02-01 05:30:00,30,Fear,2018-02-01,05:30:00
1,2018-02-02 05:30:00,15,Extreme Fear,2018-02-02,05:30:00
2,2018-02-03 05:30:00,40,Fear,2018-02-03,05:30:00
3,2018-02-04 05:30:00,24,Extreme Fear,2018-02-04,05:30:00
4,2018-02-05 05:30:00,11,Extreme Fear,2018-02-05,05:30:00


In [ ]:
# ----------------------------
# Clean Trades Dataset
# ----------------------------


# Standardize column names
trades_df.columns=trades_df.columns.str.lower().str.strip().str.replace(' ','_')

# Convert both timestamp columns
trades_df['timestamp_ist'] = pd.to_datetime(trades_df['timestamp_ist'], format='%d-%m-%Y %H:%M', errors='coerce')

# If 'timestamp' is a UNIX-like integer
if np.issubdtype(trades_df['timestamp'].dtype, np.number):
    trades_df['timestamp'] = pd.to_datetime(trades_df['timestamp'], unit='s', errors='coerce')
else:
    trades_df['timestamp'] = pd.to_datetime(trades_df['timestamp'], errors='coerce')

# Compare to see if they’re essentially the same
time_diff = (trades_df['timestamp_ist'] - trades_df['timestamp']).dropna().dt.total_seconds().abs().mean()
print(f"Average time difference between timestamp_ist and timestamp: {time_diff:.2f} seconds")

# The difference is negligible so drop 'timestamp' and use 'timestamp_ist', since it has timezone context

trades_df.drop(columns=['timestamp'], inplace=True)
print("Dropped 'timestamp' — kept 'timestamp_ist' as the main time reference.")

# Add date & time-only columns for flexibility
trades_df['date'] = trades_df['timestamp_ist'].dt.date
trades_df['time_only'] = trades_df['timestamp_ist'].dt.time

# Create a date column for merging
trades_df['date']=trades_df['timestamp_ist'].dt.date



Average time difference between timestamp_ist and timestamp: nan seconds
Dropped 'timestamp' — kept 'timestamp_ist' as the main time reference.


In [496]:
# Check for missing values
print("Null values in trades dataset:")
print(trades_df.isna().sum())

Null values in trades dataset:
account             0
coin                0
execution_price     0
size_tokens         0
size_usd            0
side                0
timestamp_ist       0
start_position      0
direction           0
closed_pnl          0
transaction_hash    0
order_id            0
crossed             0
fee                 0
trade_id            0
date                0
time_only           0
dtype: int64


In [ ]:
# Convert numeric columns safely
numeric_cols = ['execution_price', 'size_tokens', 'size_usd', 'closed_pnl','start_position','fee'] 
for col in numeric_cols:
    trades_df[col] = pd.to_numeric(trades_df[col], errors='coerce')


(trades_df[['execution_price', 'size_tokens', 'size_usd']] <= 0).sum()

#If USD value is 0, it’s almost always an invalid trade.
trades_df=trades_df[trades_df['size_usd']>0]            # keep only positive values

trades_df['is_profitable'] = (trades_df['closed_pnl'] > 0).astype(int)


trades_df['crossed'] = trades_df['crossed'].astype(str).str.strip().str.lower().map({'true': True, 'false': False})


# Clean categorical/text columns
text_cols = ['side', 'direction']
for col in text_cols:
    trades_df[col] = trades_df[col].astype(str).str.strip().str.upper()

# If both are same, drop one
if (trades_df['side'] == trades_df['direction']).all():
    trades_df.drop(columns=['direction'], inplace=True)



In [498]:
# Drop duplicates and reset index
trades_df.drop_duplicates(inplace=True)
trades_df.reset_index(drop=True, inplace=True)

print("\n✅ Trader dataset cleaned and ready for merge.")
print(trades_df.info())
display(trades_df.head())


✅ Trader dataset cleaned and ready for merge.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211181 entries, 0 to 211180
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   account           211181 non-null  object        
 1   coin              211181 non-null  object        
 2   execution_price   211181 non-null  float64       
 3   size_tokens       211181 non-null  float64       
 4   size_usd          211181 non-null  float64       
 5   side              211181 non-null  object        
 6   timestamp_ist     211181 non-null  datetime64[ns]
 7   start_position    211181 non-null  float64       
 8   direction         211181 non-null  object        
 9   closed_pnl        211181 non-null  float64       
 10  transaction_hash  211181 non-null  object        
 11  order_id          211181 non-null  int64         
 12  crossed           211181 non-null  bool          
 13  fee         

,account,coin,execution_price,size_tokens,size_usd,side,timestamp_ist,start_position,direction,closed_pnl,transaction_hash,order_id,crossed,fee,trade_id,date,time_only,is_profitable
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,BUY,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,2024-12-02,22:50:00,0
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,BUY,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,2024-12-02,22:50:00,0
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,2024-12-02 22:50:00,1002.518996,BUY,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,2024-12-02,22:50:00,0
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,2024-12-02 22:50:00,1146.558564,BUY,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,2024-12-02,22:50:00,0
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,2024-12-02 22:50:00,1289.488521,BUY,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,2024-12-02,22:50:00,0


In [499]:
# Now merge using 'date' as the key
merged_df = pd.merge(trades_df, sentiment_df[['date', 'classification']], on='date', how='left')

# Optional: Drop identifiers
merged_df.drop(columns=['account', 'transaction_hash', 'order_id', 'trade_id'], inplace=True)

print("✅ Merge successful!")
print("Merged dataset shape:", merged_df.shape)
display(merged_df.head())


✅ Merge successful!
Merged dataset shape: (211181, 15)


,coin,execution_price,size_tokens,size_usd,side,timestamp_ist,start_position,direction,closed_pnl,crossed,fee,date,time_only,is_profitable,classification
0,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,BUY,0.0,True,0.345404,2024-12-02,22:50:00,0,Extreme Greed
1,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,BUY,0.0,True,0.005600,2024-12-02,22:50:00,0,Extreme Greed
2,@107,7.9855,144.09,1150.63,BUY,2024-12-02 22:50:00,1002.518996,BUY,0.0,True,0.050431,2024-12-02,22:50:00,0,Extreme Greed
3,@107,7.9874,142.98,1142.04,BUY,2024-12-02 22:50:00,1146.558564,BUY,0.0,True,0.050043,2024-12-02,22:50:00,0,Extreme Greed
4,@107,7.9894,8.73,69.75,BUY,2024-12-02 22:50:00,1289.488521,BUY,0.0,True,0.003055,2024-12-02,22:50:00,0,Extreme Greed


In [500]:
for cols in merged_df.columns:
    if merged_df[cols].isna().sum()!=0:
        print(cols, merged_df[cols].isna().sum())

classification 6


Since we merged on 'date', that means 6 dates that are present on trades dataset did not exist in the sentiment dataset.

Let’s check which dates have NaN sentiment:

In [501]:
# Check missing sentiment
merged_df.loc[merged_df['classification'].isna(), ['date']].drop_duplicates()


,date
727,2024-10-26


So trades happened on 2024-10-26, but the sentiment dataset does not contain that date.

This can happen because:

- The sentiment index wasn’t published that day (weekend / outage).

- Or the timestamp alignment (timezone difference) shifted the date by a few hours.



Sentiment doesn’t change dramatically from one day to the next.

So we can use the most recent known sentiment (previous day’s value).

In [502]:
# Sort to ensure proper chronological fill
merged_df = merged_df.sort_values('date')
# Handle NaN sentiment
merged_df['classification'] = merged_df['classification'].ffill()

# Verify after fixing
print("Remaining NaN in classification:", merged_df['classification'].isna().sum())

Remaining NaN in classification: 0


Let pnl_pct denote the percentage return (or loss) on each trade 

The profit percentage is given by

$$
\text{pnl\_pct} = \frac{\text{closed\_pnl}}{\text{size\_usd}} \times 100
$$


In [503]:
# Create pnl_pct
merged_df['pnl_pct'] = merged_df['closed_pnl'] / merged_df['size_usd']


#### Frequency of sentiment labels

Let's check sentiment distribution

In [504]:
print(merged_df['classification'].value_counts())


classification
Fear             61826
Greed            50289
Extreme Greed    39980
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64


➡️ This means the dataset is well-balanced, though “Fear” days dominate slightly.

#### Basic statistics per sentiment group

Now let us do the summary stats grouped by sentiment to understand average performance and trading behavior during “Fear” vs “Greed”.

This shows whether traders tend to be more or less profitable during Fear vs Greed days.***

In [505]:
# Grouped summary by sentiment
grouped = merged_df.groupby('classification').agg(
    n_trades = ('pnl_pct','size'),
    mean_closed_pnl = ('closed_pnl','mean'),
    median_closed_pnl = ('closed_pnl','median'),
    mean_pnl_pct = ('pnl_pct','mean'),
    win_rate = ('is_profitable','mean'),
    avg_size_usd = ('size_usd','mean'),
    avg_execution_price = ('execution_price','mean'),
    avg_fee = ('fee','mean'),
).round(4).sort_values(('mean_pnl_pct'), ascending=False)
display(grouped)


,n_trades,mean_closed_pnl,median_closed_pnl,mean_pnl_pct,win_rate,avg_size_usd,avg_execution_price,avg_fee
classification,,,,,,,,
Extreme Greed,39980,67.9132,0.0,0.0400,0.4651,3113.1857,6084.0214,0.6761
Greed,50289,43.6000,0.0,0.0197,0.3851,5740.2447,13415.0102,1.2549
Fear,61826,54.3001,0.0,0.0154,0.4208,7817.5006,14155.1382,1.4954
Neutral,37686,34.3077,0.0,0.0099,0.3970,4782.7327,12393.6928,1.0448
Extreme Fear,21400,34.5379,0.0,0.0043,0.3706,5349.7318,7054.7951,1.1163



💰 Profitability (mean_closed_pnl & pnl%)

- Extreme Greed traders earned the highest mean PnL (≈ 67.9 USD) and highest mean PnL% (0.04%).

- Extreme Fear and Neutral had the lowest returns and win rates (~0.37–0.40).

- Median PnL = 0 for all, suggesting most trades are small wins/losses.

Interpretation:

- Traders tend to take higher risks (and larger profits) during Extreme Greed phases.

- During Fear, they reduce position size and profitability.